In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/mumbai_cleaned.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (18846, 26)


,Title,Price,Area,BHK,Bathrooms,Furnished,Balconies,Floor,Ownership,Facing,...,Description,Property Image,Price_per_m2,Suspicious_Price,Price_num,Area_num,BHK_num,Bathrooms_num,Balconies_num,Furnished_clean
0,1BHK Multistorey Apartment for Rent in Bandra ...,70000.0,750.0,1.0,1.0,Furnished,NaN,7.0,Unknown,South -West,...,"1 BHK, Multistorey Apartment is available for ...",67249079_2_PropertyImage345-4855396928822.jpg,93.333333,False,70000.0,750.0,1.0,1.0,NaN,Furnished
1,3BHK Multistorey Apartment for Rent in Lodha N...,140000.0,1980.0,3.0,4.0,Unfurnished,4.0,21.0,Unknown,East,...,"3 BHK, Multistorey Apartment is available for ...",66155783_5_hatsAppImage20230217at13.56.28.jpeg,70.707071,False,140000.0,1980.0,3.0,4.0,4.0,Unfurnished
2,3BHK Multistorey Apartment for Rent in Lodha P...,190000.0,1592.0,3.0,3.0,Unfurnished,1.0,20.0,Unknown,North - East,...,"3 BHK, Multistorey Apartment is available for ...",66686709_1_HOTO20230407165614.jpg,119.346734,False,190000.0,1592.0,3.0,3.0,1.0,Unfurnished
3,3BHK Multistorey Apartment for Rent in Rustomj...,110000.0,1305.0,3.0,3.0,Unfurnished,3.0,11.0,Freehold,West,...,"3 BHK, Multistorey Apartment is available for ...",67875387_1_PropertyImage559-8385323621333.jpg,84.291188,False,110000.0,1305.0,3.0,3.0,3.0,Unfurnished
4,3BHK Multistorey Apartment for Rent in DLH The...,140000.0,1350.0,3.0,3.0,Unfurnished,NaN,29.0,Freehold,East,...,"3 BHK, Multistorey Apartment is available for ...",66670931_3_PropertyImage314-6685675359874.jpg,103.703704,False,140000.0,1350.0,3.0,3.0,NaN,Unfurnished


Can we predict monthly rental prices in Mumbai based on property characteristics?

In [2]:
print("Dataset shape:", df.shape)

print("\nAvailable columns:")
for col in df.columns:
    print(col)

print("\nColumn types:")
display(df.dtypes)

Dataset shape: (18846, 26)

Available columns:
Title
Price
Area
BHK
Bathrooms
Furnished
Balconies
Floor
Ownership
Facing
Amenities
Transaction Type
Property Type
Location
Year of Construction
Is Luxury
Description
Property Image
Price_per_m2
Suspicious_Price
Price_num
Area_num
BHK_num
Bathrooms_num
Balconies_num
Furnished_clean

Column types:


Title                    object
Price                   float64
Area                    float64
BHK                     float64
Bathrooms               float64
Furnished                object
Balconies               float64
Floor                   float64
Ownership                object
Facing                   object
Amenities                object
Transaction Type         object
Property Type            object
Location                 object
Year of Construction    float64
Is Luxury                object
Description              object
Property Image           object
Price_per_m2            float64
Suspicious_Price           bool
Price_num               float64
Area_num                float64
BHK_num                 float64
Bathrooms_num           float64
Balconies_num           float64
Furnished_clean          object
dtype: object

In [3]:
print(df.columns.tolist())

['Title', 'Price', 'Area', 'BHK', 'Bathrooms', 'Furnished', 'Balconies', 'Floor', 'Ownership', 'Facing', 'Amenities', 'Transaction Type', 'Property Type', 'Location', 'Year of Construction', 'Is Luxury', 'Description', 'Property Image', 'Price_per_m2', 'Suspicious_Price', 'Price_num', 'Area_num', 'BHK_num', 'Bathrooms_num', 'Balconies_num', 'Furnished_clean']


## 3. Train/Test Split

We split the dataset into 80% training data and 20% test data.

The training set will be used for model training, cross-validation,
and hyperparameter tuning.

The test set will remain untouched until the final evaluation
to assess how well the model generalizes to unseen properties.

In [5]:
from sklearn.model_selection import train_test_split

# Define the target and selected features
target = "Price_num"

features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num",
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

# Verify that all columns exist
assert all(col in df.columns for col in features + [target])

# Keep only rows with a valid target
model_df = df.dropna(subset=[target]).copy()

X = model_df[features]
y = model_df[target]

# Split the dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (15076, 8)
Testing features: (3770, 8)
Training target: (15076,)
Testing target: (3770,)


## 3. Preprocessing Pipeline

Numerical and categorical features require different preprocessing.

- Numerical features: median imputation and standardization.
- Categorical features: most-frequent imputation and one-hot encoding.

All preprocessing is fitted only on training data through a
scikit-learn pipeline to prevent data leakage.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical and categorical columns
numeric_features = [
    "Area_num",
    "BHK_num",
    "Bathrooms_num",
    "Balconies_num"
]

categorical_features = [
    "Furnished_clean",
    "Facing",
    "Location",
    "Property Type"
]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


Underfitting and Overfitting Analysis
We investigated underfitting and overfitting using a Decision Tree Regressor to predict monthly rental prices in Mumbai. We varied the maximum depth of the decision tree to examine how model complexity affected prediction accuracy.

We compared training and validation RMSE and R² scores for each model. A model with high errors on both datasets may indicate underfitting, while a large difference between training and validation errors may indicate overfitting.

Based on these results, we selected a tree depth that offered a suitable balance between learning the training data and generalising to unseen properties. We then evaluated the selected model on the held-out test set.

Ensuring sklearn is installed and working in this branch.

In [4]:
import sklearn
print(sklearn.__version__)

from sklearn.linear_model import LinearRegression
model = LinearRegression()

print("Scikit-learn is working!")


1.9.0
Scikit-learn is working!


Importing the model for Underfitting & Overfitting

In [1]:
from sklearn.module import model
model = Model()
model.fit(X,y)
predictions = model.predict(X_new)
print(predictions)

ModuleNotFoundError: No module named 'sklearn.module'